# Imports & Config

In [ ]:
import numpy as np
import pandas as pd
import random
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns

N_RECORDS = 20000
N_VEHICLES = 2500
START_YEAR = 2015
END_YEAR = 2025
SERVICE_INTERVAL_DAYS = 180

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
np.random.seed(42)
random.seed(42)

# Static Data

In [ ]:
nissan_models = {
    "Altima": {"engine": "I4", "base_failure_rate": 0.02},
    "Sunny": {"engine": "I4", "base_failure_rate": 0.015},
    "Patrol": {"engine": "V6", "base_failure_rate": 0.035},
    "X-Trail": {"engine": "I4", "base_failure_rate": 0.025},
    "Pathfinder": {"engine": "V6", "base_failure_rate": 0.03}
}

failure_types = [
    # Engine System
    "Engine Overheating",
    "Oil Leak",
    "Spark Plug Failure",
    "Fuel Injector Failure",

    # Brake System
    "Brake Pad Wear",
    "Brake Disc Warp",
    "Brake Fluid Leak",

    # Electrical System
    "Battery Failure",
    "Alternator Failure",
    "Starter Motor Failure",

    # Transmission System
    "Clutch Wear",
    "Gearbox Failure",
    "Transmission Fluid Leak",

    # Suspension & Steering
    "Shock Absorber Wear",
    "Control Arm Failure",
    "Wheel Alignment Issue",
    "Power Steering Failure",

    # Tires & Wheels
    "Tire Wear",
    "Tire Blowout",
    "Wheel Bearing Failure",

    # Cooling System
    "Radiator Leak",
    "Coolant Pump Failure",
    "Thermostat Failure",

    # AC System
    "AC Compressor Failure",
    "Refrigerant Leak"
]

parts_map = {
    # Engine
    "Engine Overheating": ["Radiator", "Coolant Pump", "Thermostat"],
    "Oil Leak": ["Oil Seal", "Gasket"],
    "Spark Plug Failure": ["Spark Plug"],
    "Fuel Injector Failure": ["Fuel Injector"],

    # Brake
    "Brake Pad Wear": ["Brake Pads"],
    "Brake Disc Warp": ["Brake Disc"],
    "Brake Fluid Leak": ["Brake Hose", "Brake Fluid"],

    # Electrical
    "Battery Failure": ["Battery"],
    "Alternator Failure": ["Alternator"],
    "Starter Motor Failure": ["Starter Motor"],

    # Transmission
    "Clutch Wear": ["Clutch Kit"],
    "Gearbox Failure": ["Gearbox"],
    "Transmission Fluid Leak": ["Transmission Seal", "Transmission Fluid"],

    # Suspension
    "Shock Absorber Wear": ["Shock Absorber"],
    "Control Arm Failure": ["Control Arm"],
    "Wheel Alignment Issue": ["Alignment Service"],
    "Power Steering Failure": ["Power Steering Pump"],

    # Tires
    "Tire Wear": ["Tire"],
    "Tire Blowout": ["Tire"],
    "Wheel Bearing Failure": ["Wheel Bearing"],

    # Cooling
    "Radiator Leak": ["Radiator"],
    "Coolant Pump Failure": ["Coolant Pump"],
    "Thermostat Failure": ["Thermostat"],

    # AC
    "AC Compressor Failure": ["AC Compressor"],
    "Refrigerant Leak": ["Refrigerant"]
}

failure_weights = {
    # High-frequency wear items
    "Brake Pad Wear": 0.12,
    "Tire Wear": 0.10,
    "Shock Absorber Wear": 0.08,
    "Clutch Wear": 0.07,

    # Medium frequency
    "Battery Failure": 0.08,
    "Oil Leak": 0.07,
    "Wheel Alignment Issue": 0.06,
    "Brake Disc Warp": 0.05,

    # Lower frequency but common faults
    "Spark Plug Failure": 0.05,
    "Fuel Injector Failure": 0.04,
    "Alternator Failure": 0.04,
    "Radiator Leak": 0.04,

    # Rare but impactful
    "Engine Overheating": 0.03,
    "Gearbox Failure": 0.02,
    "Starter Motor Failure": 0.02,
    "Power Steering Failure": 0.02,
    "Wheel Bearing Failure": 0.02,

    # Environment-driven
    "Tire Blowout": 0.02,
    "Coolant Pump Failure": 0.02,
    "Thermostat Failure": 0.02,

    # AC-related (region dependent)
    "AC Compressor Failure": 0.02,
    "Refrigerant Leak": 0.02,

    # Rare leaks
    "Brake Fluid Leak": 0.015,
    "Transmission Fluid Leak": 0.015
}

failure_severity_map = {
    # Critical = Emergency likely
    "Engine Overheating": "High",
    "Gearbox Failure": "High",
    "Brake Fluid Leak": "High",
    "Tire Blowout": "High",
    "Power Steering Failure": "High",

    # Medium = Corrective mostly
    "Alternator Failure": "Medium",
    "Starter Motor Failure": "Medium",
    "Radiator Leak": "Medium",
    "Coolant Pump Failure": "Medium",
    "AC Compressor Failure": "Medium",

    # Low = Planned repair
    "Brake Pad Wear": "Low",
    "Tire Wear": "Low",
    "Clutch Wear": "Low",
    "Oil Leak": "Low",
    "Spark Plug Failure": "Low",
    "Wheel Alignment Issue": "Low"
}

seasons = ["Winter", "Spring", "Summer", "Autumn"]

# Helper Functions

In [ ]:
def random_date(start_year, end_year):
    start = datetime(start_year, 1, 1)
    end = datetime(end_year, 12, 31)
    delta = end - start
    return start + timedelta(days=random.randint(0, delta.days))


def get_season(date):
    month = date.month
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"


def temperature_by_season(season):
    if season == "Summer":
        return np.random.normal(42, 5)
    elif season == "Winter":
        return np.random.normal(22, 4)
    elif season == "Spring":
        return np.random.normal(30, 4)
    else:
        return np.random.normal(32, 4)

def get_weighted_failure(failure_weights, mileage, temp, vehicle_age, missed_services, driving_style):
    adjusted_weights = failure_weights.copy()

    for failure, weight in adjusted_weights.items():

        # Heat effects (Oman realism)
        if temp > 40:
            if "Battery" in failure:
                adjusted_weights[failure] *= 1.8
            if "Overheating" in failure or "Radiator" in failure:
                adjusted_weights[failure] *= 1.6
            if "Tire Blowout" in failure:
                adjusted_weights[failure] *= 1.5

        # Mileage-driven wear
        if mileage > 100000:
            if "Wear" in failure or "Brake" in failure or "Clutch" in failure:
                adjusted_weights[failure] *= 1.5

        # Age effects
        if vehicle_age > 8:
            if "Leak" in failure or "Failure" in failure:
                adjusted_weights[failure] *= 1.4

        # Missed service amplification
        if missed_services > 0:
            if "Oil" in failure or "Engine" in failure:
                adjusted_weights[failure] *= 1.6

        # Driving behavior
        if driving_style == "Aggressive":
            if "Brake" in failure or "Clutch" in failure or "Tire" in failure:
                adjusted_weights[failure] *= 1.5

    # Normalize weights
    total = sum(adjusted_weights.values())
    probs = [w / total for w in adjusted_weights.values()]

    return np.random.choice(list(adjusted_weights.keys()), p=probs)

def determine_service_type(failure_type, severity_map, temp, mileage):
    
    if failure_type == "None":
        return "Preventive"

    severity = severity_map.get(failure_type, "Medium")

    # High severity = mostly emergency
    if severity == "High":
        return np.random.choice(
            ["Emergency", "Corrective"],
            p=[0.8, 0.2]
        )

    # Medium severity → depends on conditions
    elif severity == "Medium":
        emergency_prob = 0.3

        # Heat increases urgency
        if temp > 40:
            emergency_prob += 0.2

        # Very high mileage = more catastrophic behavior
        if mileage > 150000:
            emergency_prob += 0.1

        emergency_prob = min(emergency_prob, 0.99)

        return np.random.choice(
            ["Emergency", "Corrective"],
            p=[emergency_prob, 1 - emergency_prob]
        )

    # Low severity → almost always corrective
    else:
        return np.random.choice(
            ["Corrective", "Emergency"],
            p=[0.99, 0.01]
        )

# Vehicle Registry

In [ ]:
vehicle_registry = {}

for vid in range(1, N_VEHICLES + 1):
    vehicle_id = f"V{vid}"
    
    model = random.choice(list(nissan_models.keys()))
    engine = nissan_models[model]["engine"]
    base_failure_rate = nissan_models[model]["base_failure_rate"]
    
    manufacture_year = random.randint(2008, 2022)
    
    driving_style = random.choices(
        ["Calm", "Moderate", "Aggressive"],
        weights=[0.4, 0.4, 0.2]
    )[0]

    avg_daily_km = max(5, np.random.normal(40, 10))
    
    vehicle_registry[vehicle_id] = {
        "model": model,
        "engine": engine,
        "base_failure_rate": base_failure_rate,
        "manufacture_year": manufacture_year,
        "driving_style": driving_style,
        "avg_daily_km": avg_daily_km
    }

# Generator

In [ ]:
from math import nan


records = []

for vehicle_id, v in vehicle_registry.items():

    current_date = datetime(v["manufacture_year"], 1, 1)
    end_date = datetime(END_YEAR, 12, 31)

    mileage = 0
    last_service_date = current_date
    missed_services = 0

    while current_date < end_date:
        service_intreval = int(np.random.normal(SERVICE_INTERVAL_DAYS, 15)) # Service delays (Noise)

        # Move to next scheduled service
        next_service_date = last_service_date + timedelta(days=service_intreval)

        # Advance time
        days_passed = (next_service_date - current_date).days
        mileage += int(v["avg_daily_km"] * days_passed)

        current_date = next_service_date

        season = get_season(current_date)
        temp = temperature_by_season(season)

        vehicle_age = current_date.year - v["manufacture_year"]

        # Did the vehicle actually come in for service?
        serviced = np.random.rand() > 0.2  

        if not serviced:
            missed_services += 1

        # Failure probability model 
        failure_prob = v["base_failure_rate"]
        failure_prob += vehicle_age * 0.005
        failure_prob += mileage / 200000
        failure_prob += missed_services * 0.08  # penalty for skipping

        if season == "Summer":
            failure_prob += 0.05

        if v["driving_style"] == "Aggressive":
            failure_prob += 0.05

        failure_prob = min(failure_prob, 0.99)

        failure_occurred = np.random.rand() < failure_prob

        if failure_occurred:
            failure_type = get_weighted_failure(
                failure_weights,
                mileage,
                temp,
                vehicle_age,
                missed_services,
                v["driving_style"]
            )
            service_type = determine_service_type(
                failure_type,
                failure_severity_map,
                temp,
                mileage
            )
            parts = parts_map[failure_type]
            cost = np.random.normal(400, 150) # Should be based on part

            # Reset after failure repair
            missed_services = 0
            last_service_date = current_date

        elif serviced:
            failure_type = "None"
            service_type = "Preventive"
            parts = ["Oil Filter", "Engine Oil"]
            cost = np.random.normal(120, 40)

            # Successful service resets missed count
            missed_services = 0
            last_service_date = current_date

        else:
            # Skipped service
            failure_type = "None"
            service_type = nan
            parts = ""
            cost = nan
            service_type = "Skipped"
            cost = 0

        records.append({
            "vehicle_id": vehicle_id,
            "model": v["model"],
            "engine_type": v["engine"],
            "manufacture_year": v["manufacture_year"],
            "service_date": current_date,
            "season": season,
            "ambient_temp": temp,
            "vehicle_age": vehicle_age,
            "mileage": mileage,
            "avg_daily_km": v["avg_daily_km"],
            "driving_style": v["driving_style"],
            "missed_services": missed_services,
            "service_type": service_type,
            "failure_occurred": int(failure_occurred),
            "failure_type": failure_type,
            "parts_replaced": ",".join(parts),
            "service_cost": cost,
        })

df = pd.DataFrame(records)

# Save

In [ ]:
df.to_csv("synthetic_nissan_pdm.csv", index=False)

# Analysis

In [ ]:
df = pd.read_csv("synthetic_nissan_pdm.csv")

In [ ]:
df[(df["vehicle_id"].duplicated(keep=False)) & (df["vehicle_id"] == "V457")]

In [ ]:
df[(df["missed_services"]>0) & (df["vehicle_id"]=="V2498")]

In [ ]:
df[(df["service_type"]=="Emergency")]

In [ ]:
df[(df["service_type"]=="Preventive")]

In [ ]:
df[(df["failure_type"]=="Suspension Wear")]

In [ ]:
# identify column types
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print(f"Numerical columns ({len(num_cols)}):", num_cols)
print(f"Categorical columns ({len(cat_cols)}):", cat_cols)

In [ ]:
for col in num_cols:
    fig, axes = plt.subplots(1, 1, figsize=(8, 5))
    
    # Histogram + KDE
    sns.histplot(df[col].dropna(), kde=True, ax=axes)
    axes.set_title(f"{col} Distribution")
    
    # # Boxplot
    # sns.boxplot(x=df[col], ax=axes[1])
    # axes[1].set_title(f"{col} Boxplot")
    
    plt.tight_layout()
    plt.show()

In [ ]:
for col in cat_cols:
    plt.figure(figsize=(10, 5))
    
    # limit categories for readability
    top_vals = df[col].value_counts().nlargest(20)
    
    sns.barplot(x=top_vals.values, y=top_vals.index)
    plt.title(f"{col} Top Categories")
    
    plt.tight_layout()
    plt.show()